In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

# ========== 1. Load Dataset ==========
df = pd.read_csv("customer_churn_dataset.csv")

# Ensure output folders exist
os.makedirs("plots", exist_ok=True)
os.makedirs("feature_json", exist_ok=True)

# ========== 2. Column Classification ==========
# Treat low-cardinality numeric columns as categorical
cat_cols = [col for col in df.columns if df[col].nunique() < 20 and df[col].dtype != 'float64']
num_cols = [col for col in df.columns if col not in cat_cols]

print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

# ========== 3. Correlation Analysis (Numerical Only) ==========
if 'Churn' in df.columns and 'Churn' in num_cols:
    corr_matrix = df[num_cols].corr()
    target_corr = corr_matrix['Churn'].abs().sort_values(ascending=False)
    top_features = target_corr.head(6).index.tolist()  # Top 5 + target
    top_features = [f for f in top_features if f != 'Churn'][:3]  # Top 3 excluding target
else:
    # If 'Churn' not numeric or not in dataset, pick top variance numerical cols
    top_features = df[num_cols].var().sort_values(ascending=False).head(3).index.tolist()

print("Selected Top Features:", top_features)


Numerical Columns: ['tenure_months', 'monthly_usage_hours']
Categorical Columns: ['has_multiple_devices', 'customer_support_calls', 'payment_failures', 'is_premium_plan', 'churn']
Selected Top Features: ['tenure_months', 'monthly_usage_hours']


In [7]:

# ========== 4. Rule-Based Plotting & JSON Saving ==========

def process_numerical(col):
    """Generate plots + JSON for numerical feature"""
    stats = {
        "mean": float(df[col].mean()),
        "median": float(df[col].median()),
        "mode": float(df[col].mode().iloc[0]) if not df[col].mode().empty else None,
        "std": float(df[col].std()),
        "min": float(df[col].min()),
        "max": float(df[col].max()),
        "skewness": float(df[col].skew()),
        "kurtosis": float(df[col].kurt())
    }

    # Histogram
    plt.figure(figsize=(6,4))
    sns.histplot(df[col].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {col}")
    plt.savefig(f"plots/{col}_hist.png")
    plt.close()

    # Boxplot
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[col].dropna())
    plt.title(f"Boxplot of {col}")
    plt.savefig(f"plots/{col}_box.png")
    plt.close()

    # Save JSON
    with open(f"feature_json/{col}.json", "w") as f:
        json.dump(stats, f, indent=4)

    return stats


def process_categorical(col):
    """Generate plots + JSON for categorical feature"""
    value_counts = df[col].value_counts(dropna=False).to_dict()
    percentages = (df[col].value_counts(normalize=True, dropna=False) * 100).to_dict()

    results = {
        "value_counts": value_counts,
        "percentages": percentages
    }

    # Bar plot
    plt.figure(figsize=(6,4))
    df[col].value_counts().head(10).plot(kind='bar')
    plt.title(f"Top categories in {col}")
    plt.ylabel("Count")
    plt.savefig(f"plots/{col}_bar.png")
    plt.close()

    # Pie chart
    plt.figure(figsize=(6,6))
    df[col].value_counts().head(5).plot(kind='pie', autopct='%1.1f%%')
    plt.title(f"Distribution of {col}")
    plt.ylabel("")
    plt.savefig(f"plots/{col}_pie.png")
    plt.close()

    # Save JSON
    with open(f"feature_json/{col}.json", "w") as f:
        json.dump(results, f, indent=4)

    return results


# ========== 5. Process Selected Features ==========
feature_insights = {}

for feature in top_features:
    if feature in num_cols:
        feature_insights[feature] = process_numerical(feature)
    elif feature in cat_cols:
        feature_insights[feature] = process_categorical(feature)

# Save all insights in one JSON
with open("feature_json/all_features.json", "w") as f:
    json.dump(feature_insights, f, indent=4)

print("✅ Plots & JSON files generated successfully!")


✅ Plots & JSON files generated successfully!
